In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [2]:
# En TotalCharges existen 11 registros vacíos (" ")

df["TotalCharges"] = df["TotalCharges"].replace(" ", 0) # Los reemplazamos por 0 porque corresponden a clientes nuevos (tenure = 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"]) # Convertimos la columna a numérica para poder hacer cálculos

In [3]:
# No  -> 0
# Yes -> 1

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})



In [4]:
df["Churn"].head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

In [5]:
# y = variable objetivo (lo que queremos predecir)

y = df["Churn"]

# X = variables que utilizaremos para hacer la predicción

X = df.drop(
    columns=[
        "customerID", # Identificador unico
        "Churn"       # Target
    ]
)

print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


In [6]:
X.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object

In [7]:
# Convierte variables categóricas en columnas binarias (0 y 1)

X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

X_encoded.head() 

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False


In [8]:
# Cantidad de filas y columnas después del encoding

print(X_encoded.shape)

# Lista de columnas generadas

print(X_encoded.columns.tolist())

(7043, 30)
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [9]:
# ==========================================
# 9. COMPARACIÓN ANTES VS DESPUÉS
# ==========================================

print("Columnas originales:", X.shape[1])
print("Columnas después del encoding:", X_encoded.shape[1])

Columnas originales: 19
Columnas después del encoding: 30


In [10]:
X_encoded.columns.tolist()

['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'gender_Male',
 'Partner_Yes',
 'Dependents_Yes',
 'PhoneService_Yes',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_One year',
 'Contract_Two year',
 'PaperlessBilling_Yes',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check']

## Technical Note: Churn Encoding Issue

During feature engineering, an issue occurred when encoding the target variable (`Churn`).

The following mapping was applied:

```python
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})
```

Because Jupyter notebooks preserve state between executions, running this cell multiple times caused the original values (`No`, `Yes`) to be replaced by numeric values (`0`, `1`).

When the mapping was executed again, no matching values were found, resulting in all entries becoming `NaN`.

This produced errors during the train-test split process because the target variable contained missing values.

Solution:

- Reload the dataset from the original source.
- Reapply data cleaning steps.
- Apply the target encoding only once.

Lesson learned:

Jupyter notebooks maintain variable state across executions, making it important to restart the kernel and rerun the notebook when unexpected behavior occurs.

In [11]:
X_encoded.isna().sum().sort_values(ascending=False)

SeniorCitizen                            0
tenure                                   0
MonthlyCharges                           0
TotalCharges                             0
gender_Male                              0
Partner_Yes                              0
Dependents_Yes                           0
PhoneService_Yes                         0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV_No internet service          0
StreamingTV_Yes                          0
StreamingMovies_No internet service      0
StreamingMo

In [12]:
print(X_encoded.isna().sum().sum())

0


In [13]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [14]:
df["TotalCharges"].dtype

dtype('float64')

In [15]:
print(df["Churn"].head())

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


In [16]:
print(X_encoded.shape)
print(y.shape)

(7043, 30)
(7043,)


In [17]:
processed_df = X_encoded.copy()
processed_df["Churn"] = y

processed_df.to_csv(
    "../data/processed/churn_model_dataset.csv",
    index=False
)